read from data/top_syn_list.tsv , and traslate the amino acids with data/pr.reduce4.redux, where the first column is postion, second column corrosponding aas translate to A, third to B, and fourth to C, fifth to D and save it locally

In [64]:
import pandas as pd
start_aa =1

# Load the TOP_SYN_LIST.TSV file
top_syn_list = pd.read_csv('data/top_syn_list.tsv', sep='\t')

# Load the PR.REDUCE4.REDUX file
with open('data/in.reduce4.redux', 'r') as f:
    pr_reduce4_redux = f.readlines()

# Parse PR.REDUCE4.REDUX into a dictionary
translation_dict = {}
for line in pr_reduce4_redux:
    parts = line.strip().split(' ')
    position = int(parts[0])+start_aa # Convert to 1-based index
    translation_dict[position] = {
        'A': set(parts[1]),
        'B': set(parts[2]),
        'C': set(parts[3]),
        'D': set(parts[4])
    }

# Function to translate amino acids
def translate_aa(position, aa):
    if position in translation_dict:
        for key, aa_set in translation_dict[position].items():
            if aa in aa_set:
                return key
    return None

# Apply translation to the dataframe
for col in ['wildtype_aa1', 'mutate_aa1', 'wildtype_aa2', 'mutate_aa2']:
    top_syn_list[col] = top_syn_list.apply(lambda row: translate_aa(row['pos1'] if '1' in col else row['pos2'], row[col]), axis=1)

# Save the translated dataframe
top_syn_list.to_csv('data/translated_top_syn_list.tsv', sep='\t', index=False)

# 1. read dde from file


In [65]:
import pandas as pd
# Try reading the file with space as a delimiter
df = pd.read_csv('data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])
# Split the 'Mutation' column into 'First_mutation' and 'Second_mutation'
df[['First_mutation', 'Second_mutation']] = df['Mutation'].str.split('-', expand=True)
# Display the DataFrame
print(df)


/var/folders/17/rj19bvws2qscyfjmb7m44zmm0000gn/T/ipykernel_51380/111636545.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv('data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])


           Mutation       DDE  DE_double  First_DE  Second_DE First_mutation  \
0           A1B-B2A  0.012168 -20.532304 -8.380550 -12.211337            A1B   
1           A1B-B2C -0.000908 -16.498453 -8.380550  -8.179655            A1B   
2           A1B-B2D  0.000125 -16.658839 -8.380550  -8.339867            A1B   
3           A1B-A3B  0.008785 -14.738873 -8.380550  -6.419735            A1B   
4           A1B-A3C  0.008937 -13.811957 -8.380550  -5.493224            A1B   
...             ...       ...        ...       ...        ...            ...   
310072  B262C-A263C  0.390912 -16.406621 -8.433813  -8.236725          B262C   
310073  B262C-A263D  0.391073 -16.553047 -8.433813  -8.383161          B262C   
310074  B262D-A263B  0.267580 -15.236453 -8.310417  -7.223255          B262D   
310075  B262D-A263C  0.001986 -16.321536 -8.310417  -8.236725          B262D   
310076  B262D-A263D  0.002057 -16.467874 -8.310417  -8.383161          B262D   

       Second_mutation  
0             

# 2. index each sequence by its mutations compared to in.consensus.reduce4.seq

read in from ../data/in.reduce4.seq, there are 1220 sequences, create a df, with last column as mutations: with a list of mutaiton that occored compared to ../data/in.consensus.reduce4.seq the mutations are in format for example: [D148B, C140D, ...] where D and C notes the wildtime from consensus at pos 148 and 140

In [66]:
# Read the consensus sequence
with open('data/in.consensus.reduce4.seq', 'r') as f:
    consensus_sequence = f.read().strip()

# Read the 1220 sequences
sequences = []
with open('data/in.reduce4.seq', 'r') as f:
    for line in f:
        sequences.append(line.strip())

# Create a DataFrame to store sequences and their mutations
sequence_df = pd.DataFrame({'Sequence': sequences})

# Function to identify mutations compared to the consensus sequence
def find_mutations(sequence, consensus):
    mutations = []
    for i, (seq_residue, cons_residue) in enumerate(zip(sequence, consensus), start=1):
        if seq_residue != cons_residue:
            mutations.append(f"{cons_residue}{i}{seq_residue}")
    return mutations

# Add a column for mutations
sequence_df['Mutations'] = sequence_df['Sequence'].apply(lambda seq: find_mutations(seq, consensus_sequence))
sequence_df['Mutations_count'] = sequence_df['Mutations'].apply(len)
# Display the DataFrame
print(sequence_df)

                                               Sequence  \
0     ABAAABABACBBAACBDBABBDBDBBBAAAAAABAAABDAABAABA...   
1     ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...   
2     ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...   
3     ABAAABCBACABAACBABABBDBDBBBBAACAABAAABAAABAABA...   
4     ABAAABCBABABAACBDBABBDBDBBBAAACAABAAABDAABAABA...   
...                                                 ...   
1215  ABAAABCBACABAACBABABBDBDBBBBAACAABAAABDAABAAAA...   
1216  ABAAABCBACABAACBDBABBDBDBBBAAACAABAAABDAABAAAA...   
1217  ABAAABCBACABAACBABABBDBDBBBAAAAAABAAABDAABAABA...   
1218  ABAAABCBACABAACBDBABBDBDBBBAAAAAABAAABDAABAABA...   
1219  ABAAABCBACBBAACBDBABBDBDBBBBAAABABAABBDAABAABA...   

                                              Mutations  Mutations_count  
0     [C7A, A11B, C31A, C50B, A72D, A101B, C124A, B1...               13  
1     [A11B, B21C, B25A, D119A, C122B, D125A, D148C,...               13  
2     [A11B, B21C, B25A, D119A, C122B, D125A, C140D,...           

# 3. J matrix and delta e definition

## 3.1 J matrix

In [67]:
import numpy as np

#dictionary of J matrix
J_dict = {}

# Load the J matrix from the downloaded file
J = np.load('data/J.npy')

row = 0
# Determine the largest position in the 'Mutation' column of the dataframe
max_position = max(
    int(mutation[1:-1]) for mutation in df['First_mutation'].tolist() + df['Second_mutation'].tolist()
)
# print(max_position+1)
# Update the range to use the largest position
for pos1 in range(1, max_position + 1):
    for pos2 in range(pos1 + 1, max_position + 1):
        for i, aa1 in enumerate(['A', 'B', 'C', 'D']):
            for j, aa2 in enumerate(['A', 'B', 'C', 'D']):
                col = i * 4 + j
                J_dict[(pos1, pos2, aa1, aa2)] = J[row, col]
                J_dict[(pos2, pos1, aa2, aa1)] = J[row, col]
        row += 1

print(f"Dictionary created with {len(J_dict)} entries")

Dictionary created with 1102496 entries


## 3.2 define delta e

In [68]:
# Define delta E calculation
def calculate_delta_e(position, old_amino_acid, new_amino_acid, seq, J_dict):
    # E(old_amino_acid)
    energy_old = 0
    for other_pos in range(1, max_position+1):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - 1]  # Access the first sequence in sequence_list
        energy_old += J_dict.get((position, other_pos, old_amino_acid, other_aa), 0)

    # E(new_amino_acid)
    energy_new = 0
    for other_pos in range(1, max_position+1):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - 1]  # Access the first sequence in sequence_list
        energy_new += J_dict.get((position, other_pos, new_amino_acid, other_aa), 0)

    delta_e = energy_old - energy_new
    # print(f"E({old_amino_acid}) at {position}: {energy_old}")
    # print(f"E({new_amino_acid}) at {position}: {energy_new}")
    # print(f"Delta E for {old_amino_acid}{position}{new_amino_acid}: {delta_e}")
    return delta_e

# # Example usage
# position = 140
# old_amino_acid = 'C'
# new_amino_acid = 'D'
# calculate_delta_e(position, old_amino_acid, new_amino_acid, sequence_list, J_dict)

In [69]:
# define dm12
def calculate_dm12 (pos1, old_amino_acid1, new_amino_acid1, pos2, old_amino_acid2, new_amino_acid2, seq, J_dict):
    energy_old = 0
    energy_new = 0

# old energy
    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_old += J_dict.get((pos1, pos2, old_amino_acid1, old_amino_acid2), 0)
        else:
            energy_old += J_dict.get((pos1, other_pos, old_amino_acid1, other_aa), 0)

    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1]
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            # energy_old += J_dict.get((pos2, pos1, old_amino_acid2, old_amino_acid1), 0)
            continue
        else:
            energy_old += J_dict.get((pos2, other_pos, old_amino_acid2, other_aa), 0)

# new energy
    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_new += J_dict.get((pos1, pos2, new_amino_acid1, new_amino_acid2), 0)
        else:
            energy_new += J_dict.get((pos1, other_pos, new_amino_acid1, other_aa), 0)

    for other_pos in range(1, max_position+1):
        other_aa = seq[other_pos - 1] 
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            # energy_new += J_dict.get((pos2, pos1, new_amino_acid2, new_amino_acid1), 0)
            continue
        
        else:
            energy_new += J_dict.get((pos2, other_pos, new_amino_acid2, other_aa), 0)
    
    return energy_old - energy_new




## 3.3 flip on 1220

run de on all 1220 sequences with D148B, C140D (C140D, D148B), if the consensus vs one of 1220 sequences where the DE of each mutation changes its size compare to others  ( DE -DE sign change from consensus where the sign change means the sign of de D148B- deC140D sign change), record that sequence with its mutaitons and mutation len

make this a flip function, with mutation1 and 2 ans input and output the file in data/antag_out/{mut1}_{mut2}.tsv

In [70]:
import os

def flip(mutation1, mutation2):
    # Extract position and amino acid information from the mutations
    pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
    pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

    # List to store sequences with sign change
    sequences_with_sign_change = []

    # Calculate delta E for the consensus sequence
    de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
    de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

    print("-" * 50)
    print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
    print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
    print("-" * 50)

    # Iterate through all sequences
    for index, row in sequence_df.iterrows():
        sequence = row['Sequence']
        mutations = row['Mutations']
        mutation_count = row['Mutations_count']

        # Skip sequences without the specified mutations
        if mutation1 not in mutations or mutation2 not in mutations:
            continue

        # Calculate delta E for the specified mutations
        de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
        de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)

        # Check for sign change
        if (de_mutation1 - de_mutation2) * (de_mutation1_consensus - de_mutation2_consensus) < 0:
            dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)
            min_dm1_dm2 = min(de_mutation1, de_mutation2)
            max_dm1_dm2 = max(de_mutation1, de_mutation2)
            max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2

            sequences_with_sign_change.append({
                'Sequence': sequence,
                'Mutations': mutations,
                'Mutation_count': mutation_count,
                'dm1': de_mutation1,
                'dm2': de_mutation2,
                # 'Dm1m2-max/min(dm1,dm2)': dm12 - max_divide_min_dm1_dm2,
                # 'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
                'max(Dm1m2-(dm1,dm2))': max(dm12 - max_dm1_dm2, dm12 - min_dm1_dm2)
            })

    # Create a DataFrame to store the results
    sign_change_df = pd.DataFrame(sequences_with_sign_change)

    # Sort the DataFrame by 'Dm1m2-min(dm1,dm2)' in descending order
    sign_change_df = sign_change_df.sort_values(by='max(Dm1m2-(dm1,dm2))', ascending=False)

    # Define the output directory and file path
    output_dir = 'data/flip_out'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{mutation1}_{mutation2}.tsv')

    # Save the results to a TSV file
    sign_change_df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")


## 3.4 compensate on 1220

In [71]:

def compensate(mutation1, mutation2):
    # Extract position and amino acid information from the mutations
    pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
    pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

    # List to store sequences with compensation
    sequences_with_compensation = []

    # Calculate delta E for the consensus sequence
    de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
    de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

    print("-" * 50)
    print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
    print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
    print("-" * 50)

    # Iterate through all sequences
    for index, row in sequence_df.iterrows():
        sequence = row['Sequence']
        mutations = row['Mutations']
        mutation_count = row['Mutations_count']
        

        # Skip sequences without the specified mutations
        if mutation1 not in mutations or mutation2 not in mutations:
            continue

        # Calculate delta E for the specified mutations
        de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
        de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
        dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

        # Check for compensation
        if dm12 < 0 and abs(dm12) < abs(de_mutation1) and abs(dm12) < abs(de_mutation2):
            min_dm1_dm2 = min(de_mutation1, de_mutation2)
            max_dm1_dm2 = max(de_mutation1, de_mutation2)
            max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
            sequences_with_compensation.append({
                'Sequence': sequence,
                'Mutations': mutations,
                'Mutation_count': mutation_count,
                'dm1': de_mutation1,
                'dm2': de_mutation2,
                'dm12': dm12,
                'max(Dm1m2-(dm1,dm2))': max(dm12 - max_dm1_dm2, dm12 - min_dm1_dm2)
            })

    # Create a DataFrame to store the results
    compensation_df = pd.DataFrame(sequences_with_compensation)
    compensation_df = compensation_df.sort_values(by='max(Dm1m2-(dm1,dm2))', ascending=False)

    # Define the output directory and file path
    output_dir = 'data/compensate_out'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{mutation1}_{mutation2}.tsv')

    # Save the results to a TSV file
    compensation_df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")


## 3.5 antagonistic interactions

In [72]:
import os

def antagonistic(mutation1, mutation2):
    # Extract position and amino acid information from the mutations
    pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
    pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

    # List to store sequences with antagonistic interactions
    sequences_with_antagonistic = []

    # Calculate delta E for the consensus sequence
    de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
    de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

    print("-" * 50)
    print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
    print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
    print("-" * 50)

    # Iterate through all sequences
    for index, row in sequence_df.iterrows():
        sequence = row['Sequence']
        mutations = row['Mutations']
        mutation_count = row['Mutations_count']
        
        # Skip sequences without the specified mutations
        if mutation1 not in mutations or mutation2 not in mutations:
            continue
        
        # Calculate delta E for the specified mutations
        de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
        de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
        dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

        # Check for antagonistic interaction
        if dm12 < de_mutation1 and dm12 < de_mutation2:
            min_dm1_dm2 = min(de_mutation1, de_mutation2)
            max_dm1_dm2 = max(de_mutation1, de_mutation2)
            max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
            
            sequences_with_antagonistic.append({
                'Sequence': sequence,
                'Mutations': mutations,
                'Mutation_count': mutation_count,
                'dm1': de_mutation1,
                'dm2': de_mutation2,
                'dm12': dm12,
                # 'Dm1m2-max/min(dm1,dm2)': dm12 - max_divide_min_dm1_dm2,
                # 'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
                'max(Dm1m2-(dm1,dm2))': max(dm12 - max_dm1_dm2, dm12 - min_dm1_dm2)
            })

    # Create a DataFrame to store the results
    antagonistic_df = pd.DataFrame(sequences_with_antagonistic)

    # Sort the DataFrame by 'Dm1m2-min(dm1,dm2)' in descending order
    antagonistic_df = antagonistic_df.sort_values(by='max(Dm1m2-(dm1,dm2))', ascending=False)

    # Define the output directory and file path
    output_dir = 'data/antag_out'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{mutation1}_{mutation2}.tsv')

    # Save the results to a TSV file
    antagonistic_df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")


In [73]:
# Define the mutations
mutation1 = 'C140D'
mutation2 = 'D148B'

# Test the flip function
print("Testing flip function:")
flip(mutation1, mutation2)

# Test the compensate function
print("\nTesting compensate function:")
compensate(mutation1, mutation2)

# Test the antagonistic function
print("\nTesting antagonistic function:")
antagonistic(mutation1, mutation2)

Testing flip function:
--------------------------------------------------
Consensus delta E for C140D: -5.7286787033081055, D148B: -4.553504943847656
Consensus dm12: -1.7733994
--------------------------------------------------
Results saved to data/flip_out/C140D_D148B.tsv

Testing compensate function:
--------------------------------------------------
Consensus delta E for C140D: -5.7286787033081055, D148B: -4.553504943847656
Consensus dm12: -1.7733994
--------------------------------------------------
Results saved to data/compensate_out/C140D_D148B.tsv

Testing antagonistic function:
--------------------------------------------------
Consensus delta E for C140D: -5.7286787033081055, D148B: -4.553504943847656
Consensus dm12: -1.7733994
--------------------------------------------------
Results saved to data/antag_out/C140D_D148B.tsv
